# Instella-MoE ARC frozen evidence probe

Run this notebook on Kaggle with **Internet enabled** and an **NVIDIA T4 accelerator**. The first run is bounded to one hidden-action world and matched intact/amnesic/shuffled evidence controls. It downloads one public Instella Think checkpoint in 4-bit or 8-bit form and writes evidence to `/kaggle/working/instella_arc_results`.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

WORKING = Path('/kaggle/working')
REPO = WORKING / 'BAssist-instella-arc'

def run(*args, cwd=None):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run([str(value) for value in args], check=True, cwd=str(cwd) if cwd else None)

run(sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip')
run(sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==4.57.1', 'accelerate>=1.2,<2',
    'huggingface_hub>=0.34,<1', 'safetensors>=0.5,<1',
    'bitsandbytes>=0.45,<1', 'peft>=0.14,<1')

if REPO.exists():
    shutil.rmtree(REPO)
run('git', 'clone', '--depth', '1', '--branch', 'instella-arc',
    'https://github.com/AmeerUsman10/BAssist.git', REPO)

run(sys.executable, '-m', 'instella_arc.kaggle_runner',
    '--checkpoint', 'think',
    '--quantizations', 'int4', 'int8',
    '--profile', 'smoke',
    '--tasks', 'action',
    '--seed-base', '930000',
    '--max-context-tokens', '6144',
    '--output-dir', WORKING / 'instella_arc_results',
    cwd=REPO)

status_path = WORKING / 'instella_arc_results' / 'status.json'
print(status_path.read_text() if status_path.exists() else 'No status file was produced.')


In [ ]:
# Inspect the compact evidence summary after the run.
from pathlib import Path
import json
root = Path('/kaggle/working/instella_arc_results')
for name in ('status.json', 'frozen_benchmark.json'):
    path = root / name
    print(f'\n===== {name} =====')
    if path.exists():
        payload = json.loads(path.read_text())
        print(json.dumps(payload.get('report_summary', payload.get('summary', payload)), indent=2))
    else:
        print('missing')
